# ── Step 1: Convert SMILES to SDF ─────────────────────────────────────────
# Input: BRD4_1M_Top2000.csv (top-ranked compounds from QSAR screening)
# Output: BRD4_library.sdf (2D SDF for 3D preparation)

In [ ]:
# Install dependencies
!pip install rdkit pandas
import io
from rdkit import Chem
from rdkit.Chem import SDWriter
import pandas as pd
from google.colab import files

# Upload CSV file
print("Please upload your CSV file (BRD4_1M_Top2000.csv):")
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[csv_filename]))
print(f"✅ Loaded {len(df)} compounds.")

writer = SDWriter('BRD4_library.sdf')
n_written, n_failed = 0, 0

for _, row in df.iterrows():
    mol = Chem.MolFromSmiles(str(row['SMILES']))
    if mol is None:
        n_failed += 1
        continue
    mol.SetProp('_Name', str(row['zincid']))
    writer.write(mol)
    n_written += 1

writer.close()
print(f"✅ Done! Success: {n_written}, Failed: {n_failed}")
files.download('BRD4_library.sdf')

Please upload your CSV file (BRD4_1M_Top2000.csv):


Saving BRD4_1M_Top2000.csv to BRD4_1M_Top2000.csv
✅ Loaded 2000 compounds.
✅ Done! Success: 2000, Failed: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# ── Step 2: 3D Conformer Generation ───────────────────────────────────────
# Input: BRD4_library.sdf
# Output: BRD4_library_clean.sdf (3D, MMFF94s minimised, 1955 mol)

Protonation method: RDKit (neutralise charges) is selected over OpenBabel
(pH-aware) because OpenBabel-protonated ligands containing formal charges
(e.g. C19H27N4O+) cause PDBQT-to-SDF back-conversion failures in the
downstream docking pipeline. RDKit neutralisation produces clean SDF output
compatible with AutoDock Vina PDBQT preparation.
protonation_method = "RDKit (neutralise charges)"


In [ ]:
# =============================================================================
#  Ligand Preparation Pipeline for Molecular Docking
# =============================================================================
#  Pipeline : Sanitise -> Standardise -> Protonate -> Embed -> MMFF94s Minimise
#  Output   : Single lowest-energy 3D conformer per ligand in SDF/MOL2/PDB
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

output_filename = "crystallized_ligand"  # @param {type:"string"}

protonation_method = "RDKit (neutralise charges)"  # @param ["OpenBabel (pH-aware)", "RDKit (neutralise charges)"]
target_pH = 7.4  # @param {type:"number"}
n_conformers = 5  # @param {type:"integer"}
mmff_max_iters = 2000  # @param {type:"integer"}
output_sdf = True  # @param {type:"boolean"}
output_mol2 = False  # @param {type:"boolean"}
output_pdb = False  # @param {type:"boolean"}

# ---------------------------------------------------------------------------
#  Resolve parameter values
# ---------------------------------------------------------------------------

OUTPUT_BASENAME = output_filename.strip().replace(" ", "_") + "_clean"
PROTONATION_METHOD = "openbabel" if "OpenBabel" in protonation_method else "rdkit"
TARGET_PH = target_pH
N_CONFORMERS = n_conformers
MMFF_MAX_ITERS = mmff_max_iters
OUT_SDF = output_sdf
OUT_MOL2 = output_mol2
OUT_PDB = output_pdb

if not any([OUT_SDF, OUT_MOL2, OUT_PDB]):
    raise ValueError("Please select at least one output format.")

# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependencies
# ---------------------------------------------------------------------------

import sys, subprocess

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

def _run(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for label, pip_name, import_name in [
    ("rdkit", "rdkit", "rdkit.Chem"),
    ("tqdm", "tqdm", "tqdm"),
]:
    try:
        __import__(import_name)
        print(f"  [OK]  {label:24s} already installed")
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  [OK]  {label:24s} installed")

try:
    import openbabel
    print(f"  [OK]  {'openbabel':24s} already installed")
except ImportError:
    _run(["apt-get", "install", "-y", "-q", "openbabel"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openbabel-wheel"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    print(f"  [OK]  {'openbabel':24s} installed")

print("  All dependencies ready.\n")

# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import os, time, warnings, tempfile
from copy import deepcopy
from pathlib import Path
from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import SDWriter
from openbabel import openbabel as ob

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

print("  Configuration")
print("  " + "-" * 60)
print(f"  Output basename  : {OUTPUT_BASENAME}")
print(f"  Protonation      : {PROTONATION_METHOD.upper()}"
      + (f" at pH {TARGET_PH}" if PROTONATION_METHOD == "openbabel" else ""))
selected_labels = []
if OUT_SDF:  selected_labels.append("SDF")
if OUT_MOL2: selected_labels.append("MOL2")
if OUT_PDB:  selected_labels.append("PDB")
print(f"  Output formats   : {', '.join(selected_labels)}")
print(f"  Conformers       : {N_CONFORMERS} (internal; lowest-energy kept)")
print(f"  MMFF94s max iter : {MMFF_MAX_ITERS}")

# ---------------------------------------------------------------------------
#  STEP 1 : File Upload
# ---------------------------------------------------------------------------

_print_header(1, "File Upload")
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file uploaded. Please re-run the cell.")

upload_name = list(uploaded.keys())[0]
upload_path = Path(upload_name)
ext = upload_path.suffix.lower()

if ext not in (".sdf", ".mol2", ".pdb"):
    raise ValueError(f"Unsupported format '{ext}'. Please upload .sdf, .mol2, or .pdb")

print(f"\n  Uploaded : {upload_name}  ({len(uploaded[upload_name]):,} bytes)")
print(f"  Format   : {ext.upper()}")

# ---------------------------------------------------------------------------
#  STEP 2 : File Inspection and Parsing
# ---------------------------------------------------------------------------

_print_header(2, "File Inspection and Parsing")

def load_sdf(path):
    mols = []
    supplier = Chem.SDMolSupplier(str(path), removeHs=True, sanitize=True)
    for i, mol in enumerate(supplier):
        if mol is None: continue
        name = mol.GetPropsAsDict().get("_Name", "").strip() or f"mol_{i+1:04d}"
        mol.SetProp("_Name", name)
        mols.append((mol, name))
    return mols

def load_mol2(path):
    mols = []
    raw = path.read_text()
    blocks = ["@<TRIPOS>MOLECULE" + b for b in raw.split("@<TRIPOS>MOLECULE") if b.strip()]
    for i, block in enumerate(blocks):
        mol = Chem.MolFromMol2Block(block, removeHs=True, sanitize=True)
        if mol is None: continue
        lines = block.strip().splitlines()
        name = (lines[1].strip() if len(lines) > 1 else "") or f"mol_{i+1:04d}"
        mol.SetProp("_Name", name)
        mols.append((mol, name))
    return mols

def load_pdb(path):
    mols = []
    raw = path.read_text()
    blocks = []
    if "MODEL" in raw:
        current = []
        for line in raw.splitlines():
            if line.startswith("MODEL"): current = [line]
            elif line.startswith("ENDMDL"):
                current.append(line)
                blocks.append("\n".join(current))
                current = []
            else: current.append(line)
        if not blocks: blocks = [raw]
    else: blocks = [raw]

    for i, block in enumerate(blocks):
        mol = Chem.MolFromPDBBlock(block, removeHs=True, sanitize=True)
        if mol is None: continue
        name = f"mol_{i+1:04d}"
        for line in block.splitlines():
            if line.startswith("COMPND") or line.startswith("REMARK"):
                candidate = line.split(None, 1)[-1].strip()
                if candidate:
                    name = candidate[:30]
                    break
        mol.SetProp("_Name", name)
        mols.append((mol, name))
    return mols

loaders = {".sdf": load_sdf, ".mol2": load_mol2, ".pdb": load_pdb}
raw_mols = loaders[ext](upload_path)
n_valid = len(raw_mols)
print(f"  Parseable molecules : {n_valid}")

if n_valid == 0:
    raise ValueError("No valid molecules found.")

# ---------------------------------------------------------------------------
#  STEP 3 : Ligand Preparation
# ---------------------------------------------------------------------------

_print_header(3, "Ligand Preparation")

def standardise_mol(mol):
    lfc = rdMolStandardize.LargestFragmentChooser()
    te = rdMolStandardize.TautomerEnumerator()
    mol = lfc.choose(mol)
    mol = te.Canonicalize(mol)
    return mol

def protonate_rdkit(mol):
    uc = rdMolStandardize.Uncharger()
    return uc.uncharge(mol)

def protonate_openbabel(mol, ph=7.4):
    try:
        smi = Chem.MolToSmiles(mol)
        obconv = ob.OBConversion()
        obconv.SetInAndOutFormats("smi", "smi")
        obmol = ob.OBMol()
        obconv.ReadString(obmol, smi)
        obmol.AddHydrogens(False, True, ph)
        out_smi = obconv.WriteString(obmol).strip()
        out_mol = Chem.MolFromSmiles(out_smi)
        if out_mol is None: return mol
        out_mol.SetProp("_Name", mol.GetPropsAsDict().get("_Name", ""))
        return out_mol
    except: return mol

def embed_and_minimise(mol, n_confs, max_iters):
    params = rdDistGeom.ETKDGv3()
    params.randomSeed = 42
    mol3d = Chem.AddHs(mol)
    conf_ids = AllChem.EmbedMultipleConfs(mol3d, numConfs=n_confs, params=params)
    if not conf_ids: AllChem.EmbedMolecule(mol3d, AllChem.ETKDG())

    results = AllChem.MMFFOptimizeMoleculeConfs(mol3d, mmffVariant="MMFF94s", maxIters=max_iters)
    if not results: return mol3d, False

    energies = [(r[1], i) for i, r in enumerate(results) if r[0] == 0]
    best_conf = min(energies)[1] if energies else 0

    mol_out = deepcopy(mol3d)
    for cid in reversed(range(mol_out.GetNumConformers())):
        if cid != best_conf: mol_out.RemoveConformer(cid)
    return mol_out, bool(energies)

prepared = []
failed = []

for idx, (raw_mol, mol_name) in enumerate(tqdm(raw_mols, desc="  Preparing")):
    try:
        mol = Chem.AddHs(raw_mol)
        Chem.SanitizeMol(mol)
        mol = standardise_mol(mol)

        if PROTONATION_METHOD == "openbabel":
            mol = protonate_openbabel(mol, ph=TARGET_PH)
        else:
            mol = protonate_rdkit(mol)

        mol = Chem.AddHs(mol)
        mol3d, converged = embed_and_minimise(mol, n_confs=N_CONFORMERS, max_iters=MMFF_MAX_ITERS)

        mol3d.SetProp("_Name", mol_name)
        mol3d.SetProp("Converged", str(converged))
        prepared.append((mol3d, mol_name))
    except Exception as e:
        failed.append((mol_name, str(e)))

# ---------------------------------------------------------------------------
#  STEP 4 & 5 : Summary & Output
# ---------------------------------------------------------------------------

_print_header(4, "Summary & Writing Files")
print(f"  Success: {len(prepared)} | Failed: {len(failed)}")

if OUT_SDF:
    out_path = f"/content/{OUTPUT_BASENAME}.sdf"
    with SDWriter(out_path) as w:
        for m, n in prepared: w.write(m)
    print(f"  [OK] Saved SDF to {out_path}")
    files.download(out_path)

print("\n" + "=" * 72)
print("  PIPELINE COMPLETE - RDKIT NEUTRALISATION MODE")
print("=" * 72)

  STEP 0 | Installing Dependencies
  [OK]  rdkit                    installed
  [OK]  tqdm                     already installed
  [OK]  openbabel                installed
  All dependencies ready.

  Configuration
  ------------------------------------------------------------
  Output basename  : BRD4_library_clean
  Protonation      : RDKIT
  Output formats   : SDF
  Conformers       : 5 (internal; lowest-energy kept)
  MMFF94s max iter : 2000

  STEP 1 | File Upload
  Accepted formats : .sdf  |  .mol2  |  .pdb
  Single-ligand and multi-ligand files both supported.



Saving BRD4_library.sdf to BRD4_library.sdf

  Uploaded : BRD4_library.sdf  (4,243,950 bytes)
  Format   : .SDF

  STEP 2 | File Inspection and Parsing
  Parseable molecules : 2000
  Detected mode       : MULTI-LIGAND

  STEP 3 | Ligand Preparation
  Per-molecule pipeline
  ------------------------------------------------------------
  [a] Sanitise mol graph
  [b] Largest fragment selection
  [c] Canonical tautomer
  [d] Protonation         : RDKit neutralise
  [e] Add explicit hydrogens
  [f] Stereocenter audit  : flag unspecified, keep as-is
  [g] ETKDGv3 embedding   : 5 conformers (internal)
  [h] MMFF94s minimisation: lowest-energy conformer retained



  Preparing: 100%|██████████████████████████████| 2000/2000 [58:14<00:00,  1.75s/mol]



  STEP 4 | Preparation Summary
  Total input molecules    : 2000
  Successfully prepared    : 1955
  Failed                   : 45
  MMFF94s fully converged  : 1955 / 1955

  Failed molecules:
    [  36] mol_0036                        ->  Bad Conformer Id
    [  37] mol_0037                        ->  Bad Conformer Id
    [  63] mol_0063                        ->  Bad Conformer Id
    [  82] mol_0082                        ->  Bad Conformer Id
    [ 143] mol_0143                        ->  Bad Conformer Id
    [ 148] mol_0148                        ->  Bad Conformer Id
    [ 170] mol_0170                        ->  Bad Conformer Id
    [ 234] mol_0234                        ->  Bad Conformer Id
    [ 352] mol_0352                        ->  Bad Conformer Id
    [ 378] mol_0378                        ->  Bad Conformer Id
    [ 400] mol_0400                        ->  Bad Conformer Id
    [ 465] mol_0465                        ->  Bad Conformer Id
    [ 471] mol_0471                   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  PIPELINE COMPLETE
  Input format     : .SDF
  Output basename  : BRD4_library_clean
  Protonation      : RDKIT
  Output formats   : SDF
  Conformers       : 5 generated internally per ligand
                     MMFF94s minimised; lowest-energy conformer retained

  Docking compatibility
  ------------------------------------------------------------
  SDF  -> Vina (via prepare_ligand -> PDBQT), GNINA, GOLD, Glide
  MOL2 -> GOLD, DOCK6, Glide
  PDB  -> PyMOL / VMD / Chimera visualisation
